In [1]:
import pandas as pd

# Load the already-prepared patient-level dataset
df = pd.read_excel(r"D:\CTS_ML\patient_overall_risk_dataset.xlsx")

print("ML dataset loaded successfully")
print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 records:")
display(df.head())

ML dataset loaded successfully
Rows: 8973
Columns: 26

Columns:
['Patient_ID', 'Age', 'Sex', 'Years_Observed', 'First_Year', 'Last_Year', 'Average_HCC_Risk', 'Maximum_HCC_Risk', 'Latest_HCC_Risk', 'Total_Mapped_HCCs', 'Total_No_HCC_Mappings', 'Total_Failed_Mappings', 'Maximum_Diagnoses', 'Maximum_Chronic_Conditions', 'Maximum_HCC_Categories', 'Total_Encounters', 'Total_Claims', 'Total_Diagnosis_Frequency', 'Maximum_Specialist_Encounters', 'Ever_Hospitalized', 'Maximum_Repeated_Diagnosis', 'Maximum_Recent_Encounters', 'Best_Recency_Days', 'HCC_Risk_Change', 'Overall_Risk_Score', 'Risk_Level']

First 5 records:


,Patient_ID,Age,Sex,Years_Observed,First_Year,Last_Year,Average_HCC_Risk,Maximum_HCC_Risk,Latest_HCC_Risk,Total_Mapped_HCCs,...,Total_Claims,Total_Diagnosis_Frequency,Maximum_Specialist_Encounters,Ever_Hospitalized,Maximum_Repeated_Diagnosis,Maximum_Recent_Encounters,Best_Recency_Days,HCC_Risk_Change,Overall_Risk_Score,Risk_Level
0,PT000001,66,M,4,2020,2025,0.512250,0.832,0.571,4,...,26,28,2,1,3,2,0,-0.2430,47.79,MEDIUM
1,PT000002,55,M,3,2022,2025,0.579500,1.282,0.466,3,...,17,17,3,0,2,4,15,-0.0240,46.16,MEDIUM
2,PT000003,50,F,3,2020,2025,0.340000,0.340,0.340,0,...,12,12,3,0,3,2,27,0.0000,22.40,LOW
3,PT000004,44,M,5,2020,2025,0.400700,0.683,0.154,7,...,27,27,3,1,3,2,28,-0.2645,40.39,MEDIUM
4,PT000005,72,M,4,2022,2025,0.573125,1.514,0.396,2,...,23,19,3,0,2,5,10,0.0000,52.39,MEDIUM


In [2]:
# ============================================================
# CELL 2: SEPARATE FEATURES AND TARGET
# ============================================================

# ------------------------------------------------------------
# Target variable
# ------------------------------------------------------------
# This is the overall patient risk score we want the ML model
# to predict.

target_column = "Overall_Risk_Score"

# ------------------------------------------------------------
# Columns that should NOT be used as ML features
# ------------------------------------------------------------
# Patient_ID is only an identifier.
# Overall_Risk_Score is the target.
# Risk_Level is derived from the target, so using it would
# cause data leakage.

columns_to_remove = [
    "Patient_ID",
    "Overall_Risk_Score",
    "Risk_Level"
]

# ------------------------------------------------------------
# Create feature dataset
# ------------------------------------------------------------

X = df.drop(
    columns=columns_to_remove
)

# ------------------------------------------------------------
# Create target
# ------------------------------------------------------------

y = df[target_column]

# ------------------------------------------------------------
# Display information
# ------------------------------------------------------------

print("ML Dataset")
print("----------")

print("Feature rows   :", X.shape[0])
print("Feature columns:", X.shape[1])

print("\nTarget:")
print(target_column)

print("\nTarget statistics:")
print(y.describe().round(2))

print("\nFeature columns:")
print(X.columns.tolist())

ML Dataset
----------
Feature rows   : 8973
Feature columns: 23

Target:
Overall_Risk_Score

Target statistics:
count    8973.00
mean       50.01
std        15.08
min         9.61
25%        39.39
50%        50.58
75%        60.85
max        90.80
Name: Overall_Risk_Score, dtype: float64

Feature columns:
['Age', 'Sex', 'Years_Observed', 'First_Year', 'Last_Year', 'Average_HCC_Risk', 'Maximum_HCC_Risk', 'Latest_HCC_Risk', 'Total_Mapped_HCCs', 'Total_No_HCC_Mappings', 'Total_Failed_Mappings', 'Maximum_Diagnoses', 'Maximum_Chronic_Conditions', 'Maximum_HCC_Categories', 'Total_Encounters', 'Total_Claims', 'Total_Diagnosis_Frequency', 'Maximum_Specialist_Encounters', 'Ever_Hospitalized', 'Maximum_Repeated_Diagnosis', 'Maximum_Recent_Encounters', 'Best_Recency_Days', 'HCC_Risk_Change']


In [3]:
# ============================================================
# CELL 3: TRAIN / VALIDATION / TEST SPLIT
# ============================================================

from sklearn.model_selection import train_test_split

# ------------------------------------------------------------
# First split:
# 80% Training
# 20% Temporary
# ------------------------------------------------------------

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

# ------------------------------------------------------------
# Second split:
# 10% Validation
# 10% Testing
# ------------------------------------------------------------

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42
)

# ------------------------------------------------------------
# Display split sizes
# ------------------------------------------------------------

print("Dataset Split")
print("-------------")

print("Training rows   :", len(X_train))
print("Validation rows :", len(X_val))
print("Testing rows    :", len(X_test))

print("\nTotal rows:", len(X_train) + len(X_val) + len(X_test))

# ------------------------------------------------------------
# Display target statistics for each split
# ------------------------------------------------------------

print("\nTarget Statistics")
print("-----------------")

print("\nTraining:")
print(y_train.describe().round(2))

print("\nValidation:")
print(y_val.describe().round(2))

print("\nTesting:")
print(y_test.describe().round(2))

Dataset Split
-------------
Training rows   : 7178
Validation rows : 897
Testing rows    : 898

Total rows: 8973

Target Statistics
-----------------

Training:
count    7178.00
mean       50.10
std        15.07
min         9.61
25%        39.53
50%        50.75
75%        60.93
max        90.80
Name: Overall_Risk_Score, dtype: float64

Validation:
count    897.00
mean      49.25
std       14.63
min       11.15
25%       39.08
50%       49.94
75%       59.27
max       89.08
Name: Overall_Risk_Score, dtype: float64

Testing:
count    898.00
mean      50.00
std       15.56
min       11.96
25%       39.08
50%       50.22
75%       61.59
max       88.18
Name: Overall_Risk_Score, dtype: float64


In [4]:
# ============================================================
# CELL 4: LINEAR REGRESSION BASELINE
# ============================================================

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# ------------------------------------------------------------
# Identify categorical and numerical columns
# ------------------------------------------------------------

categorical_features = ["Sex"]

numerical_features = [
    column for column in X.columns
    if column not in categorical_features
]

# ------------------------------------------------------------
# Preprocessing
# ------------------------------------------------------------

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        ),
        (
            "numerical",
            StandardScaler(),
            numerical_features
        )
    ]
)

# ------------------------------------------------------------
# Create Linear Regression pipeline
# ------------------------------------------------------------

linear_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LinearRegression())
    ]
)

# ------------------------------------------------------------
# Train the model
# ------------------------------------------------------------

print("Training Linear Regression...")

linear_model.fit(
    X_train,
    y_train
)

print("Linear Regression training completed.")

# ------------------------------------------------------------
# Validation prediction
# ------------------------------------------------------------

linear_predictions = linear_model.predict(X_val)

# ------------------------------------------------------------
# Calculate validation metrics
# ------------------------------------------------------------

linear_mae = mean_absolute_error(
    y_val,
    linear_predictions
)

linear_rmse = np.sqrt(
    mean_squared_error(
        y_val,
        linear_predictions
    )
)

linear_r2 = r2_score(
    y_val,
    linear_predictions
)

# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("\nLinear Regression Validation Results")
print("------------------------------------")

print(f"MAE  : {linear_mae:.4f}")
print(f"RMSE : {linear_rmse:.4f}")
print(f"R²   : {linear_r2:.4f}")

# ------------------------------------------------------------
# Show example predictions
# ------------------------------------------------------------

comparison = pd.DataFrame({
    "Actual_Risk_Score": y_val.iloc[:10].values,
    "Predicted_Risk_Score": linear_predictions[:10]
})

comparison["Error"] = (
    comparison["Actual_Risk_Score"]
    - comparison["Predicted_Risk_Score"]
)

print("\nExample Predictions")
print("-------------------")

display(
    comparison.round(2)
)

Training Linear Regression...
Linear Regression training completed.

Linear Regression Validation Results
------------------------------------
MAE  : 3.5287
RMSE : 4.5151
R²   : 0.9046

Example Predictions
-------------------


,Actual_Risk_Score,Predicted_Risk_Score,Error
0,39.92,39.66,0.26
1,53.97,56.29,-2.32
2,56.32,57.15,-0.83
3,58.38,57.13,1.25
4,34.57,38.73,-4.16
5,33.69,35.68,-1.99
6,46.60,47.91,-1.31
7,65.84,66.51,-0.67
8,63.98,59.58,4.40
9,50.48,48.00,2.48


In [5]:
# ============================================================
# CELL 5: RANDOM FOREST REGRESSOR
# ============================================================

from sklearn.ensemble import RandomForestRegressor

# Create Random Forest pipeline
# The same preprocessing used for Linear Regression is reused.
random_forest_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestRegressor(
                n_estimators=300,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

# Train the model
print("Training Random Forest Regressor...")

random_forest_model.fit(
    X_train,
    y_train
)

print("Random Forest training completed.")

# ------------------------------------------------------------
# Predict Overall Risk Score on validation data
# ------------------------------------------------------------

rf_predictions = random_forest_model.predict(X_val)

# ------------------------------------------------------------
# Calculate regression metrics
# ------------------------------------------------------------

rf_mae = mean_absolute_error(
    y_val,
    rf_predictions
)

rf_rmse = np.sqrt(
    mean_squared_error(
        y_val,
        rf_predictions
    )
)

rf_r2 = r2_score(
    y_val,
    rf_predictions
)

# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("\nRandom Forest Validation Results")
print("--------------------------------")

print(f"MAE  : {rf_mae:.4f}")
print(f"RMSE : {rf_rmse:.4f}")
print(f"R²   : {rf_r2:.4f}")

# ------------------------------------------------------------
# Show sample predictions
# ------------------------------------------------------------

rf_comparison = pd.DataFrame({
    "Actual_Risk_Score": y_val.iloc[:10].values,
    "Predicted_Risk_Score": rf_predictions[:10]
})

rf_comparison["Error"] = (
    rf_comparison["Actual_Risk_Score"]
    - rf_comparison["Predicted_Risk_Score"]
)

print("\nExample Predictions")
print("-------------------")

display(
    rf_comparison.round(2)
)

Training Random Forest Regressor...
Random Forest training completed.

Random Forest Validation Results
--------------------------------
MAE  : 2.1261
RMSE : 2.6251
R²   : 0.9678

Example Predictions
-------------------


,Actual_Risk_Score,Predicted_Risk_Score,Error
0,39.92,41.39,-1.47
1,53.97,56.75,-2.78
2,56.32,54.20,2.12
3,58.38,60.51,-2.13
4,34.57,38.17,-3.60
5,33.69,31.46,2.23
6,46.60,47.15,-0.55
7,65.84,61.85,3.99
8,63.98,60.74,3.24
9,50.48,47.03,3.45


In [6]:
# ============================================================
# CELL 6: GRADIENT BOOSTING REGRESSOR
# ============================================================

from sklearn.ensemble import GradientBoostingRegressor

# ------------------------------------------------------------
# Create Gradient Boosting pipeline
# ------------------------------------------------------------

gradient_boosting_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            GradientBoostingRegressor(
                n_estimators=200,
                learning_rate=0.05,
                max_depth=3,
                random_state=42
            )
        )
    ]
)

# ------------------------------------------------------------
# Train the model
# ------------------------------------------------------------

print("Training Gradient Boosting Regressor...")

gradient_boosting_model.fit(
    X_train,
    y_train
)

print("Gradient Boosting training completed.")

# ------------------------------------------------------------
# Validation prediction
# ------------------------------------------------------------

gb_predictions = gradient_boosting_model.predict(X_val)

# ------------------------------------------------------------
# Calculate validation metrics
# ------------------------------------------------------------

gb_mae = mean_absolute_error(
    y_val,
    gb_predictions
)

gb_rmse = np.sqrt(
    mean_squared_error(
        y_val,
        gb_predictions
    )
)

gb_r2 = r2_score(
    y_val,
    gb_predictions
)

# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("\nGradient Boosting Validation Results")
print("------------------------------------")

print(f"MAE  : {gb_mae:.4f}")
print(f"RMSE : {gb_rmse:.4f}")
print(f"R²   : {gb_r2:.4f}")

# ------------------------------------------------------------
# Example predictions
# ------------------------------------------------------------

gb_comparison = pd.DataFrame({
    "Actual_Risk_Score": y_val.iloc[:10].values,
    "Predicted_Risk_Score": gb_predictions[:10]
})

gb_comparison["Error"] = (
    gb_comparison["Actual_Risk_Score"]
    - gb_comparison["Predicted_Risk_Score"]
)

print("\nExample Predictions")
print("-------------------")

display(
    gb_comparison.round(2)
)

Training Gradient Boosting Regressor...
Gradient Boosting training completed.

Gradient Boosting Validation Results
------------------------------------
MAE  : 1.3178
RMSE : 1.6630
R²   : 0.9871

Example Predictions
-------------------


,Actual_Risk_Score,Predicted_Risk_Score,Error
0,39.92,41.00,-1.08
1,53.97,53.06,0.91
2,56.32,55.78,0.54
3,58.38,60.67,-2.29
4,34.57,35.67,-1.10
5,33.69,32.07,1.62
6,46.60,46.74,-0.14
7,65.84,66.05,-0.21
8,63.98,61.00,2.98
9,50.48,49.94,0.54


In [7]:
# ============================================================
# CELL 7: COMPARE ALL REGRESSION MODELS
# ============================================================

# Create a comparison table for all three models

model_comparison = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Random Forest",
        "Gradient Boosting"
    ],
    
    "MAE": [
        linear_mae,
        rf_mae,
        gb_mae
    ],
    
    "RMSE": [
        linear_rmse,
        rf_rmse,
        gb_rmse
    ],
    
    "R2": [
        linear_r2,
        rf_r2,
        gb_r2
    ]
})

# Round the values for easy reading
model_comparison[
    ["MAE", "RMSE", "R2"]
] = model_comparison[
    ["MAE", "RMSE", "R2"]
].round(4)

print("Model Comparison")
print("----------------")

display(model_comparison)

Model Comparison
----------------


,Model,MAE,RMSE,R2
0,Linear Regression,3.5287,4.5151,0.9046
1,Random Forest,2.1261,2.6251,0.9678
2,Gradient Boosting,1.3178,1.6630,0.9871


In [8]:
# ============================================================
# CELL 8: FINAL TEST EVALUATION
# ============================================================

# ------------------------------------------------------------
# Use the selected Gradient Boosting model
# on the completely untouched test dataset.
# ------------------------------------------------------------

final_predictions = gradient_boosting_model.predict(X_test)

# ------------------------------------------------------------
# Calculate final test metrics
# ------------------------------------------------------------

final_mae = mean_absolute_error(
    y_test,
    final_predictions
)

final_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        final_predictions
    )
)

final_r2 = r2_score(
    y_test,
    final_predictions
)

# ------------------------------------------------------------
# Display final results
# ------------------------------------------------------------

print("FINAL TEST RESULTS")
print("------------------")

print(f"MAE  : {final_mae:.4f}")
print(f"RMSE : {final_rmse:.4f}")
print(f"R²   : {final_r2:.4f}")

# ------------------------------------------------------------
# Compare actual vs predicted risk scores
# ------------------------------------------------------------

final_comparison = pd.DataFrame({
    "Actual_Risk_Score": y_test.iloc[:20].values,
    "Predicted_Risk_Score": final_predictions[:20]
})

final_comparison["Absolute_Error"] = (
    abs(
        final_comparison["Actual_Risk_Score"]
        -
        final_comparison["Predicted_Risk_Score"]
    )
)

print("\nFinal Test Predictions")
print("----------------------")

display(
    final_comparison.round(2)
)

FINAL TEST RESULTS
------------------
MAE  : 1.3216
RMSE : 1.6553
R²   : 0.9887

Final Test Predictions
----------------------


,Actual_Risk_Score,Predicted_Risk_Score,Absolute_Error
0,36.39,34.45,1.94
1,54.63,53.06,1.57
2,52.65,50.05,2.60
3,58.10,59.42,1.32
4,48.59,51.16,2.57
5,30.66,32.27,1.61
6,27.72,28.27,0.55
7,70.24,68.79,1.45
8,55.09,54.76,0.33
9,21.63,21.27,0.36


In [10]:
# ============================================================
# CELL 9: CREATE DATA-DRIVEN RISK THRESHOLDS
# ============================================================

# Calculate thresholds ONLY from training data
LOW_THRESHOLD = y_train.quantile(0.25)
HIGH_THRESHOLD = y_train.quantile(0.75)

print("Risk Thresholds")
print("---------------")

print(f"LOW threshold  : {LOW_THRESHOLD:.2f}")
print(f"HIGH threshold : {HIGH_THRESHOLD:.2f}")

print("\nRisk Level Rules")
print("----------------")

print(f"LOW    : Score < {LOW_THRESHOLD:.2f}")
print(
    f"MEDIUM : {LOW_THRESHOLD:.2f} <= Score <= {HIGH_THRESHOLD:.2f}"
)
print(f"HIGH   : Score > {HIGH_THRESHOLD:.2f}")

Risk Thresholds
---------------
LOW threshold  : 39.53
HIGH threshold : 60.93

Risk Level Rules
----------------
LOW    : Score < 39.53
MEDIUM : 39.53 <= Score <= 60.93
HIGH   : Score > 60.93


In [11]:
# ============================================================
# CELL 10: CONVERT PREDICTED RISK SCORE TO RISK LEVEL
# ============================================================

def get_risk_level(score):

    if score < LOW_THRESHOLD:
        return "LOW"

    elif score <= HIGH_THRESHOLD:
        return "MEDIUM"

    else:
        return "HIGH"


# Convert model predictions into risk levels
predicted_risk_levels = [
    get_risk_level(score)
    for score in final_predictions
]


# Create final prediction results
test_results = pd.DataFrame({
    "Actual_Risk_Score": y_test.values,
    "Predicted_Risk_Score": final_predictions,
    "Predicted_Risk_Level": predicted_risk_levels
})


# Convert actual scores into their corresponding risk levels
test_results["Actual_Risk_Level"] = (
    test_results["Actual_Risk_Score"]
    .apply(get_risk_level)
)


print("Final Risk Predictions")
print("----------------------")

display(
    test_results.head(20).round(2)
)


print("\nPredicted Risk Level Distribution")
print("---------------------------------")

print(
    test_results["Predicted_Risk_Level"]
    .value_counts()
)

Final Risk Predictions
----------------------


,Actual_Risk_Score,Predicted_Risk_Score,Predicted_Risk_Level,Actual_Risk_Level
0,36.39,34.45,LOW,LOW
1,54.63,53.06,MEDIUM,MEDIUM
2,52.65,50.05,MEDIUM,MEDIUM
3,58.10,59.42,MEDIUM,MEDIUM
4,48.59,51.16,MEDIUM,MEDIUM
5,30.66,32.27,LOW,LOW
6,27.72,28.27,LOW,LOW
7,70.24,68.79,HIGH,HIGH
8,55.09,54.76,MEDIUM,MEDIUM
9,21.63,21.27,LOW,LOW



Predicted Risk Level Distribution
---------------------------------
Predicted_Risk_Level
MEDIUM    441
LOW       230
HIGH      227
Name: count, dtype: int64


In [12]:
# ============================================================
# CELL 11: RISK LEVEL AGREEMENT
# ============================================================

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ------------------------------------------------------------
# Compare actual and predicted risk levels
# ------------------------------------------------------------

actual_levels = test_results["Actual_Risk_Level"]
predicted_levels = test_results["Predicted_Risk_Level"]

# ------------------------------------------------------------
# Calculate category agreement
# ------------------------------------------------------------

risk_level_accuracy = accuracy_score(
    actual_levels,
    predicted_levels
)

print("Risk Level Agreement")
print("--------------------")

print(
    f"Category Agreement : {risk_level_accuracy:.4f}"
)

print(
    f"Category Agreement : {risk_level_accuracy * 100:.2f}%"
)

# ------------------------------------------------------------
# Classification report
# ------------------------------------------------------------

print("\nRisk Level Classification Report")
print("--------------------------------")

print(
    classification_report(
        actual_levels,
        predicted_levels,
        labels=["LOW", "MEDIUM", "HIGH"],
        zero_division=0
    )
)

# ------------------------------------------------------------
# Confusion matrix
# ------------------------------------------------------------

print("\nRisk Level Confusion Matrix")
print("---------------------------")

cm = confusion_matrix(
    actual_levels,
    predicted_levels,
    labels=["LOW", "MEDIUM", "HIGH"]
)

cm_df = pd.DataFrame(
    cm,
    index=["Actual LOW", "Actual MEDIUM", "Actual HIGH"],
    columns=["Predicted LOW", "Predicted MEDIUM", "Predicted HIGH"]
)

display(cm_df)

Risk Level Agreement
--------------------
Category Agreement : 0.9499
Category Agreement : 94.99%

Risk Level Classification Report
--------------------------------
              precision    recall  f1-score   support

         LOW       0.97      0.95      0.96       234
      MEDIUM       0.94      0.96      0.95       430
        HIGH       0.96      0.93      0.95       234

    accuracy                           0.95       898
   macro avg       0.95      0.95      0.95       898
weighted avg       0.95      0.95      0.95       898


Risk Level Confusion Matrix
---------------------------


,Predicted LOW,Predicted MEDIUM,Predicted HIGH
Actual LOW,222,12,0
Actual MEDIUM,8,413,9
Actual HIGH,0,16,218


In [13]:
# ============================================================
# CELL 12: INDIVIDUAL PATIENT RISK PREDICTION
# ============================================================

# ------------------------------------------------------------
# Select one patient from the test dataset
# ------------------------------------------------------------

sample_patient = X_test.iloc[[0]]

# Get the actual risk score for comparison
actual_score = y_test.iloc[0]

# ------------------------------------------------------------
# Predict Overall Risk Score
# ------------------------------------------------------------

predicted_score = gradient_boosting_model.predict(
    sample_patient
)[0]

# ------------------------------------------------------------
# Convert predicted score to risk level
# ------------------------------------------------------------

predicted_level = get_risk_level(
    predicted_score
)

# ------------------------------------------------------------
# Display result
# ------------------------------------------------------------

print("Individual Patient Risk Prediction")
print("----------------------------------")

print(f"Actual Risk Score    : {actual_score:.2f}")
print(f"Predicted Risk Score : {predicted_score:.2f}")
print(f"Predicted Risk Level : {predicted_level}")

print("\nRisk Thresholds")
print("---------------")

print(f"LOW    : Score < {LOW_THRESHOLD:.2f}")
print(
    f"MEDIUM : {LOW_THRESHOLD:.2f} - {HIGH_THRESHOLD:.2f}"
)
print(f"HIGH   : Score > {HIGH_THRESHOLD:.2f}")

Individual Patient Risk Prediction
----------------------------------
Actual Risk Score    : 36.39
Predicted Risk Score : 34.45
Predicted Risk Level : LOW

Risk Thresholds
---------------
LOW    : Score < 39.53
MEDIUM : 39.53 - 60.93
HIGH   : Score > 60.93


In [14]:
# ============================================================
# CELL 13: SAVE MODEL AS PKL
# ============================================================

import joblib

# ------------------------------------------------------------
# Package everything required for prediction
# ------------------------------------------------------------

model_package = {
    "model": gradient_boosting_model,
    "feature_columns": X.columns.tolist(),
    "low_threshold": LOW_THRESHOLD,
    "high_threshold": HIGH_THRESHOLD,
    "target": "Overall_Risk_Score"
}

# ------------------------------------------------------------
# Save as PKL
# ------------------------------------------------------------

model_path = "patient_risk_model.pkl"

joblib.dump(
    model_package,
    model_path
)

print("Model saved successfully!")
print(f"PKL file: {model_path}")

Model saved successfully!
PKL file: patient_risk_model.pkl


In [15]:
# ============================================================
# CELL 14: LOAD AND VERIFY PKL
# ============================================================

import joblib

# ------------------------------------------------------------
# Load saved model package
# ------------------------------------------------------------

loaded_package = joblib.load(
    "patient_risk_model.pkl"
)

# Extract components
loaded_model = loaded_package["model"]
loaded_features = loaded_package["feature_columns"]
loaded_low_threshold = loaded_package["low_threshold"]
loaded_high_threshold = loaded_package["high_threshold"]

# ------------------------------------------------------------
# Predict using the loaded PKL
# ------------------------------------------------------------

loaded_prediction = loaded_model.predict(
    sample_patient
)[0]

# ------------------------------------------------------------
# Determine risk level
# ------------------------------------------------------------

if loaded_prediction < loaded_low_threshold:
    loaded_risk_level = "LOW"

elif loaded_prediction <= loaded_high_threshold:
    loaded_risk_level = "MEDIUM"

else:
    loaded_risk_level = "HIGH"

# ------------------------------------------------------------
# Display verification
# ------------------------------------------------------------

print("PKL Verification")
print("----------------")

print("PKL loaded successfully!")
print(f"Predicted Risk Score : {loaded_prediction:.2f}")
print(f"Predicted Risk Level : {loaded_risk_level}")

print("\nFeature count stored in PKL:")
print(len(loaded_features))

print("\nTarget stored in PKL:")
print(loaded_package["target"])

PKL Verification
----------------
PKL loaded successfully!
Predicted Risk Score : 34.45
Predicted Risk Level : LOW

Feature count stored in PKL:
23

Target stored in PKL:
Overall_Risk_Score
